In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

sys.path.append(os.path.abspath("../"))

from util import dataset as u_dataset
from util import metrics as u_metrics

In [ ]:
log_path = Path("../../logs/fit/")

In [ ]:
# Okabe-Ito color palette
okabe_ito_colors = [
    "#E69F00",  # Orange
    "#56B4E9",  # Sky Blue
    "#009E73",  # Bluish Green
    "#F0E442",  # Yellow
    "#0072B2",  # Blue
    "#D55E00",  # Vermillion
    "#CC79A7",  # Reddish Purple
    "#000000",  # Black
]


def plot_pr_curve(
    mode: str,
    object_name: str,
    specification_strings: list[str],
    threshold: float = None,
):
    base_dir = Path(log_path, mode)
    folders = [f for f in base_dir.iterdir() if f.is_dir()]
    if len(folders) != 1:
        raise ValueError(f"Expected exactly one folder in {base_dir}, found {len(folders)}")
    path_to_model = folders[0]

    # Load data for all distances
    all_recalls = []
    all_precisions = []
    candidate_strings = []
    for spec in specification_strings:
        recalls = np.load(
            path_to_model.as_posix() + f"/evaluation/{spec}/recalls/{object_name}.npy"
        )
        precisions = np.load(
            path_to_model.as_posix() + f"/evaluation/{spec}/precisions/{object_name}.npy"
        )
        all_recalls.append(recalls)
        all_precisions.append(precisions)

        n_candidates = spec.split("_")[-1].split("-")
        if object_name == u_dataset.CategoryNames.BALL.value:
            candidate_strings.append(n_candidates[0])
        elif object_name == u_dataset.CategoryNames.PENALTYMARK.value:
            candidate_strings.append(n_candidates[1])
        elif u_dataset.CategoryNames.INTERSECTIONS.value in object_name:
            candidate_strings.append(n_candidates[2])

    plot(all_recalls, all_precisions, candidate_strings, object_name, threshold)


def plot(
    all_recalls: list[np.ndarray],
    all_precisions: list[np.ndarray],
    candidate_strings: list[str],
    object_name: str,
    threshold: float = None,
):
    fig, ax = plt.subplots(figsize=(6, 4.5))

    # Define the threshold range
    threshold_range = np.linspace(0, 1, num=100)

    for i, (recalls, precisions, spec) in enumerate(
        zip(all_recalls, all_precisions, candidate_strings, strict=False)
    ):
        # Plot Precision vs. Threshold

        # threshold_range = threshold_range[1:]
        # precisions = precisions[::-1]
        # recalls = recalls

        beta = 0.6
        # f_beta = (1 + beta**2) * (precisions * recalls) / (beta**2 * precisions + recalls)
        # optimal_idx = np.argmax(f_beta)
        # optimal_threshold = threshold_range[optimal_idx]
        
        optimal_threshold, f_beta = u_metrics.calculate_optimal_threshold(beta, precisions, recalls, threshold_range)

        ax.plot(
            threshold_range,
            f_beta,
            color=okabe_ito_colors[-1],
            linewidth=2,
            linestyle="-",
            label=r"$F_\beta$" if i == 0 else "",
        )

        ax.plot(
            threshold_range,
            precisions,
            color=okabe_ito_colors[0],
            linewidth=2,
            linestyle="-",
            label="Precision" if i == 0 else "",
        )

        # Plot Recall vs. Threshold
        ax.plot(
            threshold_range,
            recalls,
            color=okabe_ito_colors[1],
            linewidth=2,
            linestyle="-",
            label="Recall" if i == 0 else "",
        )

    # Draw a red vertical line at the specified threshold
    if threshold is not None:
        ax.axvline(
            x=optimal_threshold,
            color="red",
            linestyle="dotted",
            linewidth=2,
            label=r"Schwellwert $\omega_c^*$" + f"= {optimal_threshold:.2f}",
        )

    # ax.set_title(object_name)
    ax.set_xlabel("Schwellwert", fontsize=13)
    ax.set_ylabel("Value", fontsize=13)
    ax.set_xlim([0.01, 1.0])
    ax.set_ylim([0.0, 1.0])
    ax.legend(loc="lower left", fontsize=11)
    ax.grid(True, linestyle="--", alpha=0.5)

    plt.tight_layout()
    save_path = "../../plots/threshold_determination/"
    os.makedirs(save_path, exist_ok=True)
    # plt.savefig(f"{save_path}{object_name}.pdf")
    plt.show()

In [ ]:
def plot_final_curves(
    mode, distances, n_candidates, categories, threshold, intersection_types=None
):
    specification_strings = []
    for distance in distances:
        for n_candidate in n_candidates:
            specification_strings.append(f"d_{distance}-K_{n_candidate}")

    for object in categories:
        if object == u_dataset.CategoryNames.INTERSECTIONS:
            for type in intersection_types:
                object_name = f"{object.value}_{type.name}"
                plot_pr_curve(mode, object_name, specification_strings, threshold)

        else:
            plot_pr_curve(mode, object.value, specification_strings, threshold)


In [ ]:
categories = [
    u_dataset.CategoryNames.BALL,
    # u_dataset.CategoryNames.PENALTYMARK,
    # u_dataset.CategoryNames.INTERSECTIONS,
]

timestamp = "20260711-000000"
mode = "final"
n_candidates = ["4-4-11"]
distances = ["9"]
threshold = 0.4
plot_final_curves(mode, distances, n_candidates, categories, threshold)

In [ ]:
categories = [
    # u_dataset.CategoryNames.BALL,
    u_dataset.CategoryNames.PENALTYMARK,
    # u_dataset.CategoryNames.INTERSECTIONS,
]

timestamp = "20260711-000000"
mode = "final"
n_candidates = ["4-4-11"]
distances = ["9"]
threshold = 0.96
plot_final_curves(mode, distances, n_candidates, categories, threshold)

In [ ]:
categories = [
    # u_dataset.CategoryNames.BALL,
    # u_dataset.CategoryNames.PENALTYMARK,
    u_dataset.CategoryNames.INTERSECTIONS,
]
intersection_types = [
    u_dataset.IntersectionType.L,
    u_dataset.IntersectionType.T,
    u_dataset.IntersectionType.X,
]

timestamp = "20260711-000000"
mode = "final"
n_candidates = ["4-4-11"]
distances = ["9"]
threshold = 0.8
plot_final_curves(mode, distances, n_candidates, categories, threshold, intersection_types)